In [105]:
# Install libraries
!pip install rasterio
!pip install geopandas
# Load the data from the repository
!curl -L https://github.com/a-taylor1/DATA450_Team5/raw/6d483968b828d4212e22990ec276f16735ab5d67/landfire_data/apr26_reformatted.zip --output apr26_reformatted.zip
!unzip -qo apr26_reformatted.zip
!mv apr26_download apr26_reformatted

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1512k  100 1512k    0     0  2102k      0 --:--:-- --:--:-- --:--:-- 2102k
mv: cannot move 'apr26_download' to 'apr26_reformatted/apr26_download': Directory not empty


In [106]:
!ls -l
!cd ./apr26_reformatted
!pwd

total 4352
drwx------ 6 root root    4096 Apr 27 04:26 apr26_download
drwx------ 7 root root    4096 Apr 27 06:19 apr26_reformatted
-rw-r--r-- 1 root root 1548337 Apr 27 13:37 apr26_reformatted.zip
drwxrwxrwx 3 root root    4096 Apr 27 05:08 LF2024_EVC_CONUS
drwxr-xr-x 3 root root    4096 Apr 27 13:37 __MACOSX
drwxr-xr-x 1 root root    4096 Apr 16 13:28 sample_data
-rw-r--r-- 1 root root 2882201 Apr 27 02:05 test_data.zip
/content


In [107]:
import rasterio
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from geopandas import read_file

%matplotlib inline

In [108]:
DATA_PATH = './apr26_reformatted'

In [109]:
layer_dirs = os.listdir(DATA_PATH)
table_dict = {}
for l in layer_dirs:
  file_list = os.listdir(f'{DATA_PATH}/{l}')
  for f in file_list:
    if f.endswith('.dbf'):
      df = read_file(f'{DATA_PATH}/{l}/{f}')
      df = df.drop(columns = ['R', 'G', 'B', 'RED', 'GREEN', 'BLUE'])
      # Ensure column names are standard strings before renaming
      df.columns = [str(col) for col in df.columns]
      df = df.set_index('VALUE')
      if 'EVH' in df.columns: df = df.drop(columns = ['EVH'])
      if 'EVT_FUEL' in df.columns: df = df.drop(columns = ['EVT_FUEL'])
      if 'fuelcover' in f: df.columns = df.columns.to_series().replace({'CLASSNAMES':'FUELCOVER'}).tolist()
      if 'fuelheight' in f: df.columns = df.columns.to_series().replace({'CLASSNAMES':'FUELHEIGHT'}).tolist()


      table_dict[f.replace('_codes.dbf', '')] = df

for k, v in table_dict.items():
  print(f' > {k}')
  display( v.head() )

 > fuelheight


,FUELHEIGHT
VALUE,
-9999,Fill-NoData
11,Open Water
12,Snow/Ice
13,Developed-Upland Deciduous Forest
14,Developed-Upland Evergreen Forest


 > fueltype


,EVT_FUEL_N
VALUE,
-9999,Fill-NoData
11,Ba Open Water
12,Ba Snow-Ice
20,Bau Urban
21,Bau Developed-Open Space


 > disturbance


,D_TYPE,D_SEVERITY,D_TIME
VALUE,,,
-9999,Fill-NoData,Fill-NoData,Fill-NoData
-1111,Fill-Not Mapped,Fill-Not Mapped,Fill-Not Mapped
0,No Disturbance,NA,NA
111,Fire,Low,One Year
112,Fire,Low,Two to Five Years


 > fuelcover


,FUELCOVER
VALUE,
-9999,Fill-NoData
11,Open Water
12,Snow/Ice
13,Developed-Upland Deciduous Forest
14,Developed-Upland Evergreen Forest


In [110]:
# Get a list of all VALUE levels
value_list = []
for k, v in table_dict.items():
  value_list.extend(v.index.tolist())
value_list = list(set(value_list))
value_list.sort()
print(f' > FOUND: {len(value_list)} CODES')

col_list = []
for k, v in table_dict.items():
  col_list.extend(v.columns)
print(f' > FOUND: {len(col_list)} COLUMNS')


 > FOUND: 1045 CODES
 > FOUND: 6 COLUMNS


In [111]:
key_table = pd.DataFrame(index = value_list, columns = col_list)
for k, v in table_dict.items():
  key_table.update(v)
key_table

,FUELHEIGHT,EVT_FUEL_N,D_TYPE,D_SEVERITY,D_TIME,FUELCOVER
-9999,Fill-NoData,Fill-NoData,Fill-NoData,Fill-NoData,Fill-NoData,Fill-NoData
-1111,NaN,NaN,Fill-Not Mapped,Fill-Not Mapped,Fill-Not Mapped,NaN
0,NaN,NaN,No Disturbance,NA,NA,NaN
11,Open Water,Ba Open Water,NaN,NaN,NaN,Open Water
12,Snow/Ice,Ba Snow-Ice,NaN,NaN,NaN,Snow/Ice
...,...,...,...,...,...,...
4627,NaN,Sh Central and Southern Polynesian Dry Coastal...,NaN,NaN,NaN,NaN
4628,NaN,Sh Central and Southern Polynesian Dry Coastal...,NaN,NaN,NaN,NaN
4800,NaN,"Tr 2815, 2817, 2821, 2899 Exotic - Short Herb",NaN,NaN,NaN,NaN
4801,NaN,"Tr 2815, 2817, 2821, 2899 Exotic - Medium Herb",NaN,NaN,NaN,NaN


In [112]:
for k, v in table_dict.items():
  print(f' > {k} has:')
  print(f'    > {len(v)} items')
  print(f'    > {sum([any(char.isdigit() for char in str(val)) for val in v.iloc[:,0]])} items with digits')

  display(v)

 > fuelheight has:
    > 56 items
    > 20 items with digits


,FUELHEIGHT
VALUE,
-9999,Fill-NoData
11,Open Water
12,Snow/Ice
13,Developed-Upland Deciduous Forest
14,Developed-Upland Evergreen Forest
15,Developed-Upland Mixed Forest
16,Developed-Upland Herbaceous
17,Developed-Upland Shrubland
18,Developed-Herbaceous Wetland Vegetation


 > fueltype has:
    > 907 items
    > 3 items with digits


,EVT_FUEL_N
VALUE,
-9999,Fill-NoData
11,Ba Open Water
12,Ba Snow-Ice
20,Bau Urban
21,Bau Developed-Open Space
...,...
4627,Sh Central and Southern Polynesian Dry Coastal...
4628,Sh Central and Southern Polynesian Dry Coastal...
4800,"Tr 2815, 2817, 2821, 2899 Exotic - Short Herb"


 > disturbance has:
    > 66 items
    > 0 items with digits


,D_TYPE,D_SEVERITY,D_TIME
VALUE,,,
-9999,Fill-NoData,Fill-NoData,Fill-NoData
-1111,Fill-Not Mapped,Fill-Not Mapped,Fill-Not Mapped
0,No Disturbance,NA,NA
111,Fire,Low,One Year
112,Fire,Low,Two to Five Years
...,...,...,...
722,Mastication,Moderate,Two to Five Years
723,Mastication,Moderate,Six to Ten Years
731,Mastication,High,One Year


 > fuelcover has:
    > 75 items
    > 35 items with digits


,FUELCOVER
VALUE,
-9999,Fill-NoData
11,Open Water
12,Snow/Ice
13,Developed-Upland Deciduous Forest
14,Developed-Upland Evergreen Forest
...,...
161,Shrub Canopy >= 10 and < 25%
162,Shrub Canopy >= 25 and < 60%
163,Shrub Canopy >= 60 and <= 100%


In [113]:
df = table_dict['fuelcover'].copy()


### **SORT CLASSNMES INTO UNIQUE COLUMNS**

In [114]:
# 1. Get all levels of CLASSNAMES
class_levels = key_table['CLASSNAMES'].to_list()

KeyError: 'CLASSNAMES'

In [ ]:
# 2. Separate continuous and discrete classes
#   NOTE: The dataframe represents continuous classes in the form "class name = xx%"
continuous_classes = []
discrete_classes = []
for c in class_levels:
  try:
    c.index('=')
    continuous_classes.append(c)
  except:
    discrete_classes.append(c)
print(f'CONTINUOUS:\n{continuous_classes[1:10]}')
print(f'CONTINUOUS:\n{discrete_classes[1:10]}')

In [ ]:
continuous_formatted = []
discrete_formatted = []
# 3. Create columns for continuous classes
def translate_classname(name:str) -> tuple[str, str]:
  split_c = [name]
  if name in continuous_classes:
    if ' = ' in name: split_c = name.split(' = ')
    elif ' >= ' in name: split_c = name.split(' >= ')
    split_c[0] = split_c[0].replace(' ', '_')
    split_c[1] = int(split_c[1].replace('%', ''))
    continuous_formatted.append(split_c[0])
  else:
    split_c[0] = name.replace(' ', '_')
    split_c.append(True)
    discrete_formatted.append(split_c[0])
  return tuple(split_c)

def extract_classname(name:str) -> str:
  return translate_classname(name)[0] # Removed .lower() to match list casings

def extract_value(name:str) -> int|bool:
  return translate_classname(name)[1]

key_table['class_name'] = key_table['CLASSNAMES'].apply(extract_classname).rename('class_name')
key_table['class_value'] = key_table['CLASSNAMES'].apply(extract_value).rename('class_value')
key_table


In [ ]:
def name_by_evcode (code:int) -> str:
  return key_table[key_table['VALUE'] == code].iloc[0]['class_name']
def value_by_evcode (code:int) -> int|bool:
  return key_table[key_table['VALUE'] == code].iloc[0]['class_value']

In [ ]:
import rasterio
import numpy as np
import pandas as pd

# 1. First, create map_df by reading the raster
with rasterio.open('LF2024_EVC_CONUS.tif') as src:
    band = src.read(1)
    transform = src.transform
    cols, rows = np.meshgrid(np.arange(src.width), np.arange(src.height))
    xs, ys = transform * (cols, rows)

map_df = pd.DataFrame({
    'x': xs.flatten(),
    'y': ys.flatten(),
    'VALUE': band.flatten()
})

# 2. Check the raster's official NoData value

# 3. Find unique values in our map dataframe
unique_map_values = map_df['VALUE'].unique()

# 4. Find which values are missing from the key_table
missing_from_key = set(unique_map_values) - set(key_table['VALUE'])

# 5. Map the raster's NoData value (32767) to the key_table's NoData value (-9999)
map_df['VALUE'] = map_df['VALUE'].replace(32767, -9999)
display(map_df.head())


In [ ]:
# Create a quick mapping from VALUE to class_name
code_to_name = dict(zip(key_table['VALUE'], key_table['class_name']))

# Map the names to a new column in map_df for fast grouping
map_df['class_name'] = map_df['VALUE'].map(code_to_name)

# Group by class_name and create a dictionary of dataframes
frames = {name: group for name, group in map_df.groupby('class_name')}

# Test with a known class name, e.g., 'open_water'
display( frames.get('open_water', pd.DataFrame()).head() )


In [ ]:
# Recreate new_df from map_df
new_df = pd.DataFrame({
    'X': map_df['x'],
    'Y': map_df['y'],
    'EVCODE': map_df['VALUE']
})

# Initialize columns based on continuous and discrete lists
for c in continuous_formatted:
    new_df[c] = 0
for c in discrete_formatted:
    new_df[c] = False

# Create dictionaries for quick mapping
code_to_name = dict(zip(key_table['VALUE'], key_table['class_name']))
code_to_value = dict(zip(key_table['VALUE'], key_table['class_value']))

# Vectorized approach: update columns in bulk for each unique EVCODE
for code in new_df['EVCODE'].unique():
    if code in code_to_name:
        col_name = code_to_name[code]
        col_val = code_to_value[code]
        # Assign the value to the corresponding column for all rows with this EVCODE
        new_df.loc[new_df['EVCODE'] == code, col_name] = col_val

display(new_df.head())


In [ ]:
# Standardize column naming
new_col_names = {}
for c in new_df.columns:
  result = c
  symbols = []
  for character in c:
    if( (not str(character).isalnum()) and (not character.isspace()) ):
      symbols.append(character)
  for s in symbols:
    result = result.replace(s, '-')
  result = [r.capitalize() for r in result.split('-') if r != '']
  result_str = ''.join(result)
  new_col_names[c] = result_str
new_df_renamed = new_df.rename(columns = new_col_names)
new_df_renamed.head()

In [ ]:
# Drop rows with no data (edge rows)
new_df_dropped = new_df_renamed[ new_df_renamed['Evcode'] != -9999]
new_df_dropped = new_df_dropped.drop(columns = ['FillNodata'])
new_df_dropped

In [ ]:
for i in new_df_dropped.dtypes.value_counts().index:
  print( f'COLUMNS OF TYPE <{i}>' )
  print( new_df_dropped.select_dtypes(str(i)).columns.values )

The dataset has bool columns and numeric columns.

In [ ]:
df_dummies = pd.get_dummies(data = new_df_dropped, prefix = '_', drop_first = True)
df_dummies.head()

In [ ]:
plt.figure(figsize=(12, 10))
sns.scatterplot(
    data = df_dummies.sample(n = int(len(df_dummies)*0.2)),
    x = 'X', y = 'Y',
    hue = 'TreeCover',
    palette = 'viridis',
    s = 2, edgecolor = None
)
plt.title('Tree Cover Map')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.show()


In [ ]:
# Get summaries of each column to identify columns with only one value
single_val_cols = []
clean_df_cols = []
for c in df_dummies.columns:
  if len(df_dummies[c].value_counts()) == 1: single_val_cols.append(c)
  else: clean_df_cols.append(c)
clean_df = df_dummies[clean_df_cols].reset_index(drop = True).drop(columns = ['Evcode'])
clean_df.head()


In [ ]:
clean_df['xStandard'] = (clean_df['X'] - clean_df['X'].min()) / (clean_df['X'].max() - clean_df['X'].min())
clean_df['yStandard'] = (clean_df['Y'] - clean_df['Y'].min()) / (clean_df['Y'].max() - clean_df['Y'].min())

clean_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def hexplot_column(column_name:str, alt_title:str = None):
  if alt_title == None: alt_title = column_name
  plt.figure(figsize=(12, 10))
  # Using C to aggregate TreeCover values with mean function
  hb = plt.hexbin(
      x = clean_df['X'],
      y = clean_df['Y'],
      C = clean_df[column_name],
      reduce_C_function = np.mean,
      gridsize = 100,
      cmap = 'Greens',
      mincnt = 1
  )
  plt.colorbar(hb, label = f'Average {alt_title}')
  plt.title(f'2D Hexbin Map of Average {alt_title}')
  plt.xlabel('X Coordinate')
  plt.ylabel('Y Coordinate')
  plt.show()

In [ ]:
hexplot_column('ShrubCover', 'Shrub Cover')

In [ ]:
hexplot_column('TreeCover', 'Tree Cover')

In [ ]:
hexplot_column('HerbCover', 'Herb Cover')

In [ ]:
clean_df.head()

